In [ ]:
import pandas as pd

# Dictionary of file paths with league names
files = {
    "Serie A": "/Users/joeyli/skillvsluck/data/european_soccer_leagues/pure_luck_goals_based/serie_a_simulated_matches_all_seeds.csv",
    "Bundesliga": "/Users/joeyli/skillvsluck/data/european_soccer_leagues/pure_luck_goals_based/bundesliga_simulated_matches_all_seeds.csv",
    "La Liga": "/Users/joeyli/skillvsluck/data/european_soccer_leagues/pure_luck_goals_based/la_liga_simulated_matches_all_seeds.csv",
    "Premier League": "/Users/joeyli/skillvsluck/data/european_soccer_leagues/pure_luck_goals_based/premier_league_simulated_matches_all_seeds.csv"
}

# Read CSVs and add 'league' column
dfs = []
for league, path in files.items():
    df = pd.read_csv(path)
    df['league'] = league
    dfs.append(df)

# Concatenate all
combined_df = pd.concat(dfs, ignore_index=True)

# Save to new CSV
combined_df.to_csv("/Users/joeyli/skillvsluck/data/european_soccer_leagues/pure_luck_goals_based/pure_luck_goals_all_leagues_combined.csv", index=False)

print("CSV files combined with league column successfully!")


In [17]:
import pandas as pd
def calculate_league_win_percentage_dispersion(in_path: str, out_path: str) -> pd.DataFrame:
    """
    Calculate the standard deviation of win percentages per league.
    
    Win percentage = total_points_in_season / (number_of_games * 3)
    Each (season, team) pair is one data point.
    Returns one standard deviation value per league.
    
    Output columns: league, std_win_percentage, num_data_points
    """
    df = pd.read_csv(in_path)
    df["home_team_points"] = df["hometeamresult"].map({1: 3, 0: 1, -1: 0})
    df["away_team_points"] = df["hometeamresult"].map({-1: 3, 0: 1, 1: 0})
    # Check required columns
    need = {"season", "league", "home_team", "away_team", "home_team_points", "away_team_points"}
    miss = need - set(df.columns)
    if miss:
        raise ValueError(f"Missing columns: {sorted(miss)}")
    
    # Create home and away dataframes
    home = pd.DataFrame({
        "league": df["league"],
        "season": df["season"],
        "team": df["home_team"],
        "points": df["home_team_points"]
    })
    
    away = pd.DataFrame({
        "league": df["league"],
        "season": df["season"],
        "team": df["away_team"],
        "points": df["away_team_points"]
    })
    
    # Combine home and away records
    long = pd.concat([home, away], ignore_index=True)
    
    # Calculate total points and games per (season, team, league)
    season_team_stats = (long
        .groupby(["league", "season", "team"], as_index=False)
        .agg(
            total_points=("points", "sum"),
            num_games=("points", "count")
        )
    )
    
    # Calculate win percentage for each (season, team) pair
    season_team_stats["win_percentage"] = (
        season_team_stats["total_points"] / (season_team_stats["num_games"] * 3)
    )
    
    # Calculate standard deviation per league across all (season, team) pairs
    league_dispersion = (season_team_stats
        .groupby("league", as_index=False)
        .agg(
            std_win_percentage=("win_percentage", lambda x: x.std(ddof=0)),
            num_data_points=("win_percentage", "count")
        )
    )
    
    # Sort by league name
    league_dispersion = league_dispersion.sort_values("league").reset_index(drop=True)
    
    # Save to CSV
    league_dispersion.to_csv(out_path, index=False)
    print(f"Wrote: {out_path} (rows: {len(league_dispersion):,})")
    
    return league_dispersion


import pandas as pd
def seed_dispersion(in_path: str, out_path: str):
    """
    Calculate the standard deviation of win percentages per league.
    
    Win percentage = total_points_in_season / (number_of_games * 3)
    Each (season, team) pair is one data point.
    Returns one standard deviation value per league.
    
    Output columns: league, std_win_percentage, num_data_points
    """

    #season,date,home_team,away_team,true_home_team_result,simulated_home_team_result_seed_1,simulated_home_team_result_seed_2,simulated_home_team_result_seed_3,simulated_home_team_result_seed_4,simulated_home_team_result_seed_5,simulated_home_team_result_seed_6,simulated_home_team_result_seed_7,simulated_home_team_result_seed_8,simulated_home_team_result_seed_9,simulated_home_team_result_seed_10,league

    for seed in range(1, 11):
        df = pd.read_csv(in_path)
        df["home_team_points"] = df[f"simulated_home_team_result_seed_{seed}"].map({1: 3, 0: 1, -1: 0})
        df["away_team_points"] = df[f"simulated_home_team_result_seed_{seed}"].map({-1: 3, 0: 1, 1: 0})
        # Check required columns
        need = {"season", "league", "home_team", "away_team", "home_team_points", "away_team_points"}
        miss = need - set(df.columns)
        if miss:
            raise ValueError(f"Missing columns: {sorted(miss)}")
        
        # Create home and away dataframes
        home = pd.DataFrame({
            "league": df["league"],
            "season": df["season"],
            "team": df["home_team"],
            "points": df["home_team_points"]
        })
        
        away = pd.DataFrame({
            "league": df["league"],
            "season": df["season"],
            "team": df["away_team"],
            "points": df["away_team_points"]
        })
        
        # Combine home and away records
        long = pd.concat([home, away], ignore_index=True)
        
        # Calculate total points and games per (season, team, league)
        season_team_stats = (long
            .groupby(["league", "season", "team"], as_index=False)
            .agg(
                total_points=("points", "sum"),
                num_games=("points", "count")
            )
        )
        
        # Calculate win percentage for each (season, team) pair
        season_team_stats["win_percentage"] = (
            season_team_stats["total_points"] / (season_team_stats["num_games"] * 3)
        )
        
        # Calculate standard deviation per league across all (season, team) pairs
        league_dispersion = (season_team_stats
            .groupby("league", as_index=False)
            .agg(
                std_win_percentage=("win_percentage", lambda x: x.std(ddof=0)),
                num_data_points=("win_percentage", "count")
            )
        )
        
        # Sort by league name
        league_dispersion = league_dispersion.sort_values("league").reset_index(drop=True)
        
        # Save to CSV
        league_dispersion.to_csv(out_path + str(seed) + ".csv" , index=False)
        print(f"Wrote: {out_path} (rows: {len(league_dispersion):,})")
    

In [ ]:
# # run to generate csv files

# Actual
actual = calculate_league_win_percentage_dispersion(
    in_path="/Users/joeyli/skillvsluck/data/european_soccer_leagues/actual/actual_combined_matches.csv",
    out_path="/Users/joeyli/skillvsluck/output/european_soccer_leagues/points_std/actual.csv",)

# Pure Skill
skill = calculate_league_win_percentage_dispersion(
    in_path="/Users/joeyli/skillvsluck/data/european_soccer_leagues/pure_skill/skill_based_league.csv",
    out_path="/Users/joeyli/skillvsluck/output/european_soccer_leagues/points_std/skilled.csv",)

# Pure Luck 
result_luck = seed_dispersion(
    in_path="/Users/joeyli/skillvsluck/data/european_soccer_leagues/pure_luck_result_based/all_leagues_combined.csv",
    out_path="/Users/joeyli/skillvsluck/data/european_soccer_leagues/pure_luck_result_based/sim",)

goals_luck = seed_dispersion(
    in_path="/Users/joeyli/skillvsluck/data/european_soccer_leagues/pure_luck_goals_based/pure_luck_goals_all_leagues_combined.csv",
    out_path="/Users/joeyli/skillvsluck/output/european_soccer_leagues/dispersion/pure_luck_goals/sim",)

Wrote: /Users/joeyli/skillvsluck/data/european_soccer_leagues/pure_luck_result_based/sim (rows: 4)
Wrote: /Users/joeyli/skillvsluck/data/european_soccer_leagues/pure_luck_result_based/sim (rows: 4)
Wrote: /Users/joeyli/skillvsluck/data/european_soccer_leagues/pure_luck_result_based/sim (rows: 4)
Wrote: /Users/joeyli/skillvsluck/data/european_soccer_leagues/pure_luck_result_based/sim (rows: 4)
Wrote: /Users/joeyli/skillvsluck/data/european_soccer_leagues/pure_luck_result_based/sim (rows: 4)
Wrote: /Users/joeyli/skillvsluck/data/european_soccer_leagues/pure_luck_result_based/sim (rows: 4)
Wrote: /Users/joeyli/skillvsluck/data/european_soccer_leagues/pure_luck_result_based/sim (rows: 4)
Wrote: /Users/joeyli/skillvsluck/data/european_soccer_leagues/pure_luck_result_based/sim (rows: 4)
Wrote: /Users/joeyli/skillvsluck/data/european_soccer_leagues/pure_luck_result_based/sim (rows: 4)
Wrote: /Users/joeyli/skillvsluck/data/european_soccer_leagues/pure_luck_result_based/sim (rows: 4)
Wrote: /Us

In [25]:
import pandas as pd
import os

# Path to your sim CSV files
folder = "/Users/joeyli/skillvsluck/output/european_soccer_leagues/dispersion/pure_luck_results/"

# Initialize an empty DataFrame
combined = None

# Loop over seeds 1 to 10
for seed in range(1, 11):
    file_path = os.path.join(folder, f"sim{seed}.csv")
    df = pd.read_csv(file_path)
    
    # Rename the std_win_percentage column to include seed
    df = df.rename(columns={"std_win_percentage": f"win_percentage_seed_{seed}"})
    
    # Keep only league and the renamed win_percentage column
    df = df[["league", f"win_percentage_seed_{seed}"]]
    
    # Merge with combined DataFrame
    if combined is None:
        combined = df
    else:
        combined = pd.merge(combined, df, on="league", how="outer")

# Optional: save combined file
combined.to_csv(os.path.join(folder, "all_seeds_combined.csv"), index=False)

print("Combined CSV created with all seeds.")


Combined CSV created with all seeds.


In [ ]:
## 
import pandas as pd

combined = pd.read_csv("/Users/joeyli/skillvsluck/output/european_soccer_leagues/dispersion/pure_luck_results/all_seeds_combined.csv")

seed_cols = [f"win_percentage_seed_{i}" for i in range(1, 11)]

combined["average_win_percentage"] = combined[seed_cols].mean(axis=1)

combined.to_csv("all_seeds_combined_with_avg.csv", index=False)
